In [19]:
# SimpleDirectoryReader is dynamic, detects file type and uses appropriate reader
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings, PromptTemplate, get_response_synthesizer, Document, StorageContext, load_index_from_storage
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.ingestion import IngestionPipeline
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from transformers import AutoTokenizer

import pandas as pd, re
from datasets import Dataset

from ragas.llms import llm_factory
from ragas.embeddings import OpenAIEmbeddings
from ragas.metrics import answer_relevancy, faithfulness, context_recall, context_precision
from ragas import evaluate
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
# from nanonets import NANONETSOCR

import torch
import os
import re
import csv

# --- For Azure ML Sandpit environment ---

# Project root path for Azure Sandpit environment
project_root_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone" 

# Change the current working directory to the project root
os.chdir(project_root_path)


# Bypass find_dotenv() and use a direct, verified path
dotenv_path = "/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env"

if not os.path.exists(dotenv_path):
    raise FileNotFoundError(
        f"CRITICAL ERROR: .env file NOT FOUND at the expected path: {dotenv_path}\n"
        f"Please double-check the path you pasted into 'project_root_path'."
    )


# -- For Local Development Environment --
load_dotenv(dotenv_path=dotenv_path)

# The project root is now simply the current working directory
project_root = os.getcwd()

# --- Now, the rest of your variable loading will work correctly ---
relative_data_dir = os.getenv("VECTOR_DATASET_DIR")

# Add a check to make sure the variable was loaded successfully from the file
if not relative_data_dir:
    raise ValueError(
        "ERROR: 'VECTOR_DATASET_DIR' was not found in your .env file, or the file is empty."
    )

data_directory = os.path.join(project_root, relative_data_dir)

hf_token = os.getenv("HUGGINGFACE_TOKEN")
llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

# --- Final Verification ---
print(f"✅ Project root successfully set to: {project_root}")
print(f"✅ .env file loaded from: {dotenv_path}")
print(f"📁 Data directory set to: {data_directory}")

✅ Project root successfully set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone
✅ .env file loaded from: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/.env
📁 Data directory set to: /mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Vector_Dataset


## Load Persistent Data

In [2]:
# Define the path where the index is saved
persist_dir = "./vector_store"

# Check if the saved index exists before trying to load it
if os.path.exists(persist_dir):
    # Load the index from the storage context
    storage_context = StorageContext.from_defaults(persist_dir=persist_dir)
    index = load_index_from_storage(storage_context)
    print(f"✅ Index successfully loaded from '{persist_dir}'.")
else:
    print(f"❌ Index not found at '{persist_dir}'.")
    print("Please run the data ingestion and saving cells first to create the index.")

# The 'index' variable is now loaded and ready for the benchmarking steps.

Loading llama_index.core.storage.kvstore.simple_kvstore from ./vector_store/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from ./vector_store/index_store.json.
✅ Index successfully loaded from './vector_store'.


## Vector/Graph Store Data Ingestion (using LlamaParse)

In [3]:
# # Check if directory exists
# if not data_directory or not os.path.isdir(data_directory):
#     raise ValueError(
#         f"The path '{data_directory}' is not a valid directory. "
#         "Please check that the VECTOR_DATASET_DIR variable is set correctly in your .env file "
#         "and that the directory actually exists."
#     )

# # Initialize LlamaParse with your API key
# llama_cloud_api_key = os.getenv("LLAMA_CLOUD_API_KEY")
# if not llama_cloud_api_key:
#     raise ValueError("LLAMA_CLOUD_API_KEY not found in your .env file. Please get a key from https://cloud.llamaindex.ai")

# parser = LlamaParse(
#     api_key=llama_cloud_api_key,
#     result_type="markdown",
#     verbose=True
# )

# # Separate the file paths based on their type (PDF vs. other)
# pdf_filepaths = []
# other_filepaths = []
# for filename in os.listdir(data_directory):
#     file_path = os.path.join(data_directory, filename)
#     if os.path.isfile(file_path):
#         if filename.lower().endswith('.pdf'):
#             pdf_filepaths.append(file_path)
#         else:
#             other_filepaths.append(file_path)

# print(f"--- Found {len(pdf_filepaths)} PDF(s) and {len(other_filepaths)} other file(s) to process. ---")

# # Process the files in batches
# all_documents = []

# # Process all PDFs in a single batch call to LlamaParse
# if pdf_filepaths:
#     print("\n- Parsing PDF files with LlamaParse...")
#     try:
#         # Calling parser.load_data() with a LIST of files is the correct way
#         pdf_docs = parser.load_data(pdf_filepaths)
#         all_documents.extend(pdf_docs)
#         print(f"  -> Successfully parsed {len(pdf_filepaths)} PDF file(s).")
#     except Exception as e:
#         print(f"  -> FAILED to parse PDFs with LlamaParse. Error: {e}")

# # Process all other files in a single batch call to SimpleDirectoryReader
# if other_filepaths:
#     print("\n- Parsing other files with SimpleDirectoryReader...")
#     try:
#         other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
#         all_documents.extend(other_docs)
#         print(f"  -> Successfully parsed {len(other_filepaths)} other file(s).")
#     except Exception as e:
#         print(f"  -> FAILED to parse other files. Error: {e}")

# # The 'documents' variable should now contain all chunks from all parsed files
# documents = all_documents
# print(f"\n--- Ingestion complete ---")
# print(f"Successfully loaded and chunked a total of {len(documents)} document(s) from all files in '{data_directory}'.")


## Vector/Graph Store Data Ingestion (Using Nanonets OCR API)

In [4]:
# # --- Helper function to safely parse Nanonets response ---

# 1. Initialize the Nanonets OCR client
# nanonets_ocr = NANONETSOCR()
# nanonets_api_key = os.getenv("NANONETS_API_KEY")
# if not nanonets_api_key:
#     raise ValueError("NANONETS_API_KEY not found in your .env file.")
# nanonets_ocr.set_token(nanonets_api_key)


# # Extracts text from JSON response from Nanonets OCR API
# def extract_text_from_nanonets_response(response, file_path):
#     """
#     A highly defensive function to safely extract text from a Nanonets OCR API response.
#     It checks for different data structures to handle both single and multi-page documents.
#     """
#     filename = os.path.basename(file_path)

#     # Check if response is a dictionary and contains 'results' key
#     # results key holds a list and each item in the list is a JSON object from one file
#     # JSON object acts as a container for the file's metadata and the extracted content by Nanonets
#     if not isinstance(response, dict) or 'results' not in response or not isinstance(response['results'], list) or not response['results']:
#         print(f" -> FAILED: {filename}. API response was malformed or empty. Response: {response}")
#         return None

#     first_result = response['results'][0]
    
#     if not isinstance(first_result, dict) or 'page_data' not in first_result:
#         print(f" -> FAILED: {filename}. 'results' object is malformed. Got: {first_result}")
#         return None

#     page_data_content = first_result['page_data']
#     full_text = ""

#     # Handle both list (multi-page) and dict (single-page) structures
#     # If file is a multi-page document it iterates through each page in list and extracts the 'raw_text'
#     # appending it to full_text with double newlines for separation
#     if isinstance(page_data_content, list):
#         for page in page_data_content:
#             if isinstance(page, dict) and 'raw_text' in page:
#                 full_text += page['raw_text'] + "\\n\\n"
#         if not full_text.strip():
#             print(f" -> FAILED: {filename}. 'page_data' list contained no extractable text.")
#             return None
    
#     # If file is a single-page document it directly extracts the 'raw_text'
#     elif isinstance(page_data_content, dict):
#         if 'raw_text' in page_data_content:
#             full_text = page_data_content['raw_text']
#         else:
#             print(f" -> FAILED: {filename}. 'page_data' dictionary missing 'raw_text' key.")
#             return None
            
#     else:
#         print(f" -> FAILED: {filename}. 'page_data' has an unexpected type: {type(page_data_content)}.")
#         return None

#     return full_text.strip()

# # Check which files need OCR processing
# ocr_filepaths = []
# other_filepaths = []
# ocr_extensions = ['.pdf', '.jpg', '.jpeg', '.png']

# for filename in os.listdir(data_directory):
#     file_path = os.path.join(data_directory, filename)
#     if os.path.isfile(file_path):
#         if any(filename.lower().endswith(ext) for ext in ocr_extensions):
#             ocr_filepaths.append(file_path)
#         else:
#             other_filepaths.append(file_path)

# # --- Process OCR files with Nanonets OCR ---
# all_documents = []

# print(f"--- Processing {len(ocr_filepaths)} file(s) with Nanonets OCR. ---")
# for file_path in ocr_filepaths:
#     try:
#         prediction = nanonets_ocr.convert_to_prediction(file_path)
        
#         doc_text = extract_text_from_nanonets_response(prediction, file_path)
        
#         # Only create a Document if text extraction was successful
#         if doc_text:
#             document = Document(text=doc_text, metadata={'file_path': file_path})
#             all_documents.append(document)
#             print(f" -> Successfully processed {os.path.basename(file_path)}")
            
#     except Exception as e:
#         print(f" -> FAILED during API call for {os.path.basename(file_path)}. An unexpected error occurred: {e}")

# # --- Process other file types with SimpleDirectoryReader ---
# if other_filepaths:
#     print(f"\\n--- Loading {len(other_filepaths)} other file(s) with SimpleDirectoryReader. ---")
#     try:
#         other_docs = SimpleDirectoryReader(input_files=other_filepaths).load_data()
#         all_documents.extend(other_docs)
#         print(f" -> Successfully parsed {len(other_docs)} other document(s).")
#     except Exception as e:
#         print(f" -> FAILED to parse other files. Error: {e}")

# documents = all_documents
# print(f"\\n--- Ingestion complete ---")
# print(f"Successfully loaded a total of {len(documents)} document(s).")



## Nanonets OCR (Local)

In [ ]:
# load_dotenv()

# model_path = os.getenv("NANONETS_OCR_S_DIR")

# # --- 3. Verify the path was loaded and is valid ---
# if not model_path or not os.path.exists(model_path):
#     raise ValueError("MODEL_PATH not found or invalid. Check your .env file and path.")
# else:
#     print(f"Model path loaded from .env: {model_path}")

# try:
#     ocr_processor = AutoProcessor.from_pretrained(model_path)
#     ocr_model = AutoModelForImageTextToText.from_pretrained(
#         model_path,
#         device_map="auto",
#         dtype=torch.bfloat16 # Corrected argument
#     )
#     print(f"✅ Nanonets OCR model ('{model_path}') loaded successfully.")
# except Exception as e:
#     print(f"❌ Failed to load Nanonets OCR model. Error: {e}")

Model path loaded from .env: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Nanonets-OCR-s


Loading checkpoint shards: 100%|██████████| 4/4 [01:17<00:00, 19.37s/it]

✅ Nanonets OCR model ('/home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Nanonets-OCR-s') loaded successfully.


In [ ]:
# all_documents = []
# print("--- Starting data ingestion using the Nanonets model directly ---")

# NANONETS_PROMPT = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""

# def ocr_page_with_nanonets(image, model, processor, max_new_tokens=4096):
#     """Correctly processes a single image page using the Nanonets VLM."""
#     messages = [
#         {"role": "user", "content": [
#             {"type": "image"},
#             {"type": "text", "text": NANONETS_PROMPT},
#         ]},
#     ]
#     text = processor.apply_chat_template(messages, add_generation_prompt=True)
#     inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt").to(model.device)
    
#     output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
#     generated_ids = [output_id[len(input_id):] for input_id, output_id in zip(inputs.input_ids, output_ids)]
    
#     output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#     return output_text[0] # Return the string from the list

# # --- Ingestion Loop ---
# try:
#     files_to_process = [f for f in os.listdir(data_directory) if os.path.isfile(os.path.join(data_directory, f))]
#     print(f"Found {len(files_to_process)} file(s) to process.")
# except (NameError, FileNotFoundError):
#     files_to_process = []

# for filename in files_to_process:
#     file_path = os.path.join(data_directory, filename)
    
#     if not filename.lower().endswith('.pdf'):
#         print(f" -> Skipping non-PDF file: {filename}")
#         continue
        
#     print(f" -> Processing PDF: {filename}...")
#     try:
#         images_from_pdf = convert_from_path(file_path, dpi=200)
#         full_text_content = ""
        
#         for i, page_image in enumerate(images_from_pdf):
#             # Pass the dedicated OCR model and processor
#             page_text = ocr_page_with_nanonets(page_image, ocr_model, ocr_processor)
#             full_text_content += page_text + "\n\n"

#         if full_text_content.strip():
#             document = Document(text=full_text_content, metadata={'file_path': file_path})
#             all_documents.append(document)
#             print(f"   - Successfully extracted content from {filename}")
#         else:
#             print(f" -> WARNING: No content was extracted from {filename}")

#     except Exception as e:
#         print(f" -> FAILED to process {filename}. Error: {e}")
#         continue

# documents = all_documents
# print(f"\n--- Ingestion complete ---")
# print(f"Successfully loaded {len(documents)} document(s).")

--- Starting data ingestion using the Nanonets model directly ---
Found 14 file(s) to process.
 -> Skipping non-PDF file: .amlignore
 -> Skipping non-PDF file: .amlignore.amltmp
 -> Skipping non-PDF file: .gitkeep
 -> Processing PDF: ANNUAL CRIME BRIEF 2020.pdf...
 -> FAILED to process ANNUAL CRIME BRIEF 2020.pdf. Error: Unable to get page count. Is poppler installed and in PATH?
 -> Processing PDF: ANNUAL CRIME BRIEF 2021.pdf...
 -> FAILED to process ANNUAL CRIME BRIEF 2021.pdf. Error: Unable to get page count. Is poppler installed and in PATH?
 -> Processing PDF: ANNUAL CRIME BRIEF 2022.pdf...
 -> FAILED to process ANNUAL CRIME BRIEF 2022.pdf. Error: Unable to get page count. Is poppler installed and in PATH?
 -> Processing PDF: Annual Crime Brief 2023.pdf...
 -> FAILED to process Annual Crime Brief 2023.pdf. Error: Unable to get page count. Is poppler installed and in PATH?
 -> Processing PDF: Annual Crime Brief 2024.pdf...
 -> FAILED to process Annual Crime Brief 2024.pdf. Error: U

## Vector/Graph Store Data Ingestion (Using SimpleDirectoryReader)

In [7]:
# Ensure data_directory points to a folder containing files (PDF, txt, md, docx, etc.)
assert os.path.isdir(data_directory), f"Data directory not found: {data_directory}"

# Use SimpleDirectoryReader to auto-detect file types and parse them
reader = SimpleDirectoryReader(input_dir=data_directory, recursive=True)
documents = reader.load_data()

print(f"Ingestion complete using SimpleDirectoryReader.")
print(f"Total documents loaded: {len(documents)}")
# Optional: peek at first doc metadata
if documents:
    print("Sample metadata:", documents[0].metadata)

Ingestion complete using SimpleDirectoryReader.
Total documents loaded: 175
Sample metadata: {'page_label': '1', 'file_name': 'ANNUAL CRIME BRIEF 2020.pdf', 'file_path': '/mnt/batch/tasks/shared/LS_root/mounts/clusters/joo-a100/code/Users/TAN_Heng_Joo/heng-joo-capstone/Datasets/Vector_Dataset/ANNUAL CRIME BRIEF 2020.pdf', 'file_type': 'application/pdf', 'file_size': 457110, 'creation_date': '2025-09-03', 'last_modified_date': '2025-09-03'}


## Llama 3.1 8B Instruct

In [8]:
load_dotenv()

model_path = os.getenv("LLAMA_3.1_8B_INSTRUCT_DIR")

# --- 3. Verify the path was loaded and is valid ---
if not model_path or not os.path.exists(model_path):
    raise ValueError("MODEL_PATH not found or invalid. Check your .env file and path.")
else:
    print(f"Model path loaded from .env: {model_path}")

# Initialize the tokenizer to get the token ID for our stop sequence
tokenizer = AutoTokenizer.from_pretrained(model_path)
# The semicolon is our desired stop character. Get its token ID.
semicolon_token_id = tokenizer.convert_tokens_to_ids(";")

# Now, initialize the LLM with the correct stop condition
llm = HuggingFaceLLM(
    model_name=model_path,
    tokenizer_name=model_path,
    device_map="auto",
    model_kwargs={"token": hf_token, "dtype": torch.bfloat16},
    # Use 'eos_token_id' which is the correct parameter for this purpose
    generate_kwargs={
        "temperature": 0.1,
        "do_sample": True,
        # This tells the model to stop generating as soon as it outputs a semicolon
        "eos_token_id": semicolon_token_id,
    }
)

print("HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.")

Model path loaded from .env: /home/azureuser/cloudfiles/code/Users/TAN_Heng_Joo/heng-joo-capstone/models/Llama-3.1-8B-Instruct


2025-09-25 04:11:43,451 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 4/4 [02:28<00:00, 37.24s/it]


HuggingFaceLLM initialized with the ';' character as the end-of-sequence token.


## Vector Embeddings

In [9]:
Settings.llm = llm
Settings.embed_model = HuggingFaceEmbedding(
    model_name="intfloat/multilingual-e5-large"
)

print("Global settings configured with Llama 3.1 and intfloat/multilingual-e5-large embedding model.")

2025-09-25 04:14:12,958 - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-large


Global settings configured with Llama 3.1 and intfloat/multilingual-e5-large embedding model.


In [10]:
# Pass the list 'documents' directly, creating separate index entries for each document chunk
# By default, VectorStoreIndex chunk size is set to 1024 characters
# And chunk overlap is set to 20% of the chunk size
# Each chunk will be 1024 characters long with a 200 character overlap
node_parser = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=200
)

pipeline = IngestionPipeline(
    transformations=[node_parser]
)

nodes = pipeline.run(documents=documents)

index = VectorStoreIndex(nodes)

print(f"Vector store index has been built successfully from {len(documents)} source document(s).")
print(f"Total nodes created with custom chunking: {len(nodes)}")
print(f"Chunk size: {node_parser.chunk_size}, Chunk overlap: {node_parser.chunk_overlap}")


Vector store index has been built successfully from 175 source document(s).
Total nodes created with custom chunking: 175
Chunk size: 1024, Chunk overlap: 200


## Retrieve Answer from Datastore

In [11]:
query_text_vector = "What percentage of reported crimes in 2020 were scams?"

# Retriever to get the top 3 most similar nodes from the index
retriever = index.as_retriever(similarity_top_k=3)
retrieved_nodes = retriever.retrieve(query_text_vector)

raw_chunks = [n.get_content() for n in retrieved_nodes]

seen = set()
cleaned_chunks = []

for txt in raw_chunks:
    # Strip known cues that cause echoing
    t = txt.replace("(1 sentence)", "").strip()

    # De-duplicate identical chunks
    if t and t not in seen:
        cleaned_chunks.append(t)
        seen.add(t)

if not cleaned_chunks:
    cleaned_chunks = [txt.replace("(1 sentence)", "").strip() for txt in raw_chunks if txt.strip()]

# Combine the content of the retrieved nodes into a single context string
# Contains the "exact answer" material for the LLM
exact_context = "\n\n---\n\n".join(cleaned_chunks)

# Check retrieved context
print("--- Retrieved Context Sent to LLM ---\n", exact_context)

conversational_prompt_template = PromptTemplate(
    "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
    "Do not invent or add anything not present in the context. Do not repeat the question. "
    "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
    "Context:\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n\n"
    "User's Question: {query_str}\n"
    "Answer: "
)

# Format the prompt with our retrieved context and the original query
final_prompt = conversational_prompt_template.format(
    context_str=exact_context,
    query_str=query_text_vector
)

# Generate with a tight token cap
raw_response = llm.complete(
    final_prompt,
    max_new_tokens=64
)

# Trim to first sentence to prevent duplication or rambling
def first_sentence(text: str) -> str:
    s = text.strip()
    # Simple split on period
    parts = s.split(".")
    return (parts[0] + ".").strip() if parts and parts else s

extracted_answer = first_sentence(str(raw_response))
print(extracted_answer)

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


--- Retrieved Context Sent to LLM ---
 Page 2 of 13 
2019. Housebreaking and related crimes decreased by 25.3% to 210 cases in 2020, 
from 281 cases in 2019. Both these crime classes recorded a 36-year low.  
 
Scam Situation in Singapore 
 
6. The total number of scam cases reported increased by 64.0% to 15,651 cases 
in 2020, from 9,545 cases in 2019. It made up 42.0% of Overall Crime in 2020, up from 
27.2% in 2019. 
 
7. Among the top ten scam types, e-commerce scams, social media impersonation 
scams, loan scams and banking related phishing scams are of particular concern, as 
they constituted 67.9% of the top ten scam types reported in 2020. Furthermore, the 
total number of reported cases for these scams increased sharply by 76.7%, compared 
to the same period in 2019. 
 
E-commerce scams remains the top scam 
 
8. E-commerce scams remains the top scam type, with the highest number of 
reported cases in 2020.   
a) E-commerce scam cases increased by 19.3% to 3,359 cases in 2020,

## Transform Extracted Answer To Be Conversational

In [12]:
# Normalize the extracted answer
# Eg "19,966." -> "19,966"
answer_core = extracted_answer.strip()
answer_core = re.sub(r"\.\s*$", "", answer_core).strip()

if not answer_core:
    print("Sorry, I couldn’t extract an answer from the context.")
else:
    REPHRASE_PROMPT = (
        "You are a precise assistant. Write exactly ONE conversational sentence that answers the question.\n"
        "Hard constraints:\n"
        "- Use the Extracted answer exactly once.\n"
        "- Do not add any other information not present in the Extracted answer or the Question.\n"
        "- Do not repeat yourself.\n"
        "- End with a single period.\n\n"
        f"Extracted answer: {answer_core}\n"
        f"Question: {query_text_vector}\n"
        "Answer:"
    )

    # Generate a short rephrased sentence to avoid loops
    rephrase_raw = llm.complete(REPHRASE_PROMPT, max_new_tokens=32)

    # Collapse whitespace
    s = " ".join(str(rephrase_raw).strip().split())

    # Remove a potential leading echo of the bare answer (e.g., "19,966. The total ...")
    if answer_core:
        s = re.sub(rf"^\s*{re.escape(answer_core)}\.\s*", "", s).strip()

    # Ensure exactly one sentence
    idx = s.find(".")
    s = (s[: idx + 1] if idx != -1 else s + ".").strip()

    # Enforce inclusion of the extracted answer exactly once
    # If the model dropped the value, fall back to the minimal guaranteed version
    if answer_core not in s:
        s = f"{answer_core}."

    print(s)

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


### Answer The percentage of reported crimes in 2020 that were scams is 42.


## Benchmarking

In [ ]:
# --- Config paths ---
relative_benchmark_path = os.getenv("VECTOR_BENCHMARK_DATASET_DIR")
if not relative_benchmark_path:
    raise ValueError("VECTOR_BENCHMARK_DATASET_DIR not set in .env")
BENCHMARK_FILE_PATH = os.path.join(project_root, relative_benchmark_path)
OUTPUT_FILENAME = "(3 SDR)base_vector_benchmark_results.csv"
OUTPUT_FILE_PATH = os.path.join(os.getcwd(), OUTPUT_FILENAME)

# --- Load and Prepare Data ---
if not os.path.isfile(BENCHMARK_FILE_PATH):
    raise FileNotFoundError(f"Benchmark file not found at: {BENCHMARK_FILE_PATH}")

# Load the benchmark CSV, automatically handling potential encoding errors
try:
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH)
except UnicodeDecodeError:
    print("UTF-8 decoding failed. Retrying with 'latin1' encoding.")
    benchmark_df = pd.read_csv(BENCHMARK_FILE_PATH, encoding='latin1')

benchmark_df['Context'] = benchmark_df['gt_context']

# --- CONFIG: Set the number of rows to test ---
# Slice to first 3 rows for a quick test
benchmark_df = benchmark_df.head(3)
print(f"Loaded {len(benchmark_df)} question-answer pairs for evaluation.")


# --- Unified Generation and Context Selection Logic ---
def get_response_and_top_context(question: str, retriever):
    """
    Retrieves context, generates a response, and returns the single top-ranked
    context chunk for clear reporting.
    """
    # Retrieve the top N nodes which are sorted by relevance
    retrieved_nodes = retriever.retrieve(question)
    
    # De-duplicate and clean all retrieved chunks for the LLM
    seen = set()
    all_cleaned_chunks = []
    for node in retrieved_nodes:
        content = node.get_content().replace("(1 sentence)", "").strip()
        if content and content not in seen:
            all_cleaned_chunks.append(content)
            seen.add(content)
            
    # The top-ranked context is simply the first one in the list.
    top_context_chunk = all_cleaned_chunks[0] if all_cleaned_chunks else ""
    
    # Combine ALL chunks into a single context pool for the LLM
    context_for_llm = "\n\n---\n\n".join(all_cleaned_chunks)

    prompt_template = PromptTemplate(
        "You are a precise Q&A assistant. Use ONLY the context to answer the single question. "
        "Do not invent or add anything not present in the context. Do not repeat the question. "
        "Do not answer any other questions. Output exactly one sentence and then stop.\n\n"
        "Context:\n"
        "---------------------\n"
        "{context_str}\n"
        "---------------------\n\n"
        "User's Question: {query_str}\n"
        "Answer: "
    )
    final_prompt = prompt_template.format(context_str=context_for_llm, query_str=question)
    
    raw_response = llm.complete(final_prompt, max_new_tokens=64)
    
    # Clean the response
    s = str(raw_response).strip()
    parts = s.split(".")
    prediction = (parts[0] + ".").strip() if parts else s
    
    # Return the prediction, all chunks for Ragas, and the top chunk for the CSV
    return prediction, all_cleaned_chunks, top_context_chunk


# --- Generate Predictions ---
ragas_data = {"question": [], "answer": [], "contexts": [], "ground_truth": []}
# Retrieve top 3 chunks to give the LLM enough context
retriever = index.as_retriever(similarity_top_k=3) 

print("\nGenerating predictions and selecting top context for each question...")
# Initialize new columns to store results
benchmark_df['response'] = ''
benchmark_df['retrieved_contexts'] = ''

for i, row in benchmark_df.iterrows():
    question = row['question'].strip()
    ground_truth = row['gt_answer'].strip()
    
    # Get the response and the single top-ranked context
    prediction, all_chunks, top_chunk = get_response_and_top_context(question, retriever)
    
    # Store results in the DataFrame
    benchmark_df.at[i, 'response'] = prediction
    # Store ONLY the top-ranked chunk for clean reporting
    benchmark_df.at[i, 'retrieved_contexts'] = top_chunk
    
    # Collect data for Ragas evaluation (Ragas still needs all chunks)
    ragas_data["question"].append(question)
    ragas_data["answer"].append(prediction)
    ragas_data["contexts"].append(all_chunks)
    ragas_data["ground_truth"].append(ground_truth)
    
    print(f"Processed {i+1}/{len(benchmark_df)}")

# --- Run Ragas Evaluation ---
ragas_dataset = Dataset.from_dict(ragas_data)
print(f"\nPrepared {len(ragas_dataset)} valid samples for Ragas evaluation.")

openai_client = OpenAI()
judge_llm = llm_factory(model="gpt-4o") 
judge_embeddings = OpenAIEmbeddings(client=openai_client, model="text-embedding-ada-002")

# Define metrics
metrics_to_evaluate = [
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision
]

print("\nRunning Ragas evaluation...")
result = evaluate(
    dataset=ragas_dataset,
    metrics=metrics_to_evaluate,
    llm=judge_llm,
    embeddings=judge_embeddings,
)
print("Ragas evaluation complete.")

# --- Format and Save Final Results ---
scores_df = result.to_pandas()

# Merge scores back into the main DataFrame
benchmark_df = benchmark_df.reset_index(drop=True)
scores_df = scores_df.reset_index(drop=True)
metric_cols = [m.name for m in metrics_to_evaluate]
benchmark_df = benchmark_df.join(scores_df[metric_cols])

# Ensure the desired column order
final_columns = ['question', 'gt_answer', 'Context', 'response', 'retrieved_contexts'] + metric_cols
benchmark_df = benchmark_df[final_columns]

# Save the final results to a CSV file
benchmark_df.to_csv(OUTPUT_FILE_PATH, index=False)
print(f"\nDetailed results saved to: {OUTPUT_FILE_PATH}")

# Print overall performance metrics
print("\n--- Overall Ragas Performance Metrics (Averages) ---")
print(scores_df[metric_cols].mean(numeric_only=True))
print("----------------------------------------------------")

Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


Loaded 3 question-answer pairs for evaluation.

Generating predictions and selecting top context for each question...


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


Processed 1/3


Setting `pad_token_id` to `eos_token_id`:26 for open-end generation.


Processed 2/3
Processed 3/3

Prepared 3 valid samples for Ragas evaluation.


TypeError: OpenAIEmbeddings.__init__() missing 1 required positional argument: 'client'